<a href="https://colab.research.google.com/github/Kevan123/TrustInAI/blob/main/Trust_Score_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trust Score AI Dashboard
### Multi-Signal, Task-Aware Trust Scoring Framework
Based on: *Operationalization of Trust in AI Outputs* — Rajaram et al.

Powered by **Groq API** (free tier — `llama-3.3-70b-versatile`, no billing required)

| Signal | Symbol | Description |
|--------|--------|-------------|
| Evidence Grounding | **E** | Are claims supported by external evidence? |
| Verifiability | **V** | Does the output agree with known ground truth? |
| Stability | **S** | Is the output robust to rephrased prompts? |
| Logical Coherence | **L** | Is the output free of contradictions/errors? |
| Calibration | **C** | Does expressed confidence match correctness? |

---
---
### This is build using Groq + Wikipedia

> **Free tier limits:** 30 requests/min · 14,400 requests/day — more than enough for this dashboard.

## Install dependencies

In [10]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

packages = ["groq", "nltk", "ipywidgets", "matplotlib", "numpy", "pandas", "requests"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed.")

✅ All dependencies installed.


## Configuration

In [11]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────────
import os


GROQ_API_KEY = ""

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Model — llama-3.3-70b-versatile is free, fast, and highly capable
# Other free options: "llama-3.1-8b-instant" (faster), "mixtral-8x7b-32768"
GROQ_MODEL = "llama-3.3-70b-versatile"

# Risk level: 0.0 = low-stakes task, 1.0 = high-stakes (medical, legal, etc.)
DEFAULT_RISK_LEVEL = 0.5

# Signal weights [E, V, S, L, C] — auto-renormalized to sum to 1
DEFAULT_WEIGHTS = [0.30, 0.30, 0.10, 0.20, 0.10]

# Routing thresholds
THRESHOLD_PUBLISH = 70   # score >= this → ✅ Publish
THRESHOLD_REVIEW  = 45   # score >= this → ⚠️  Review
                         # score <  this → 🚫 Block

print("✅ Configuration saved.")
print(f"   Model  : {GROQ_MODEL}")
print(f"   Risk   : {DEFAULT_RISK_LEVEL}")
print(f"   Weights: E={DEFAULT_WEIGHTS[0]}, V={DEFAULT_WEIGHTS[1]}, S={DEFAULT_WEIGHTS[2]}, L={DEFAULT_WEIGHTS[3]}, C={DEFAULT_WEIGHTS[4]}")

✅ Configuration saved.
   Model  : llama-3.3-70b-versatile
   Risk   : 0.5
   Weights: E=0.3, V=0.3, S=0.1, L=0.2, C=0.1


## Core Trust-Signal Functions

In [12]:
# ── Cell 3: Core Trust-Signal Functions ───────────────────────────────────────
import math, re
import numpy as np
from nltk.tokenize import sent_tokenize

# ── Helpers ───────────────────────────────────────────────────────────────────
# Stopwords for IDF-weighted containment scoring
_SCORE_STOPWORDS = {
    "is","the","a","an","of","in","on","at","to","for","and","or","but",
    "with","by","from","as","it","its","this","that","was","were","be",
    "been","have","has","yes","no","not","are","their","they","which",
    "who","also","more","most","some","such","than","then","both","each",
    "few","many","own","same","so","very","do","did","does","had","him",
    "his","her","she","he","we","us","our","you","your","they",
}

def jaccard(a: str, b: str) -> float:
    """
    IDF-weighted containment: what fraction of claim 'a' is supported by evidence 'b'?

    Replaces plain Jaccard (which penalises long evidence sentences unfairly).
    Stopwords are downweighted so content words drive the score.

    Score interpretation:
      1.0 — all content words in claim appear in evidence
      0.5 — half the content words match
      0.0 — no overlap
    """
    def _tok(text):
        return re.sub(r"[^a-z0-9\s]", "", text.lower()).split()
    def _w(t): return 0.1 if t in _SCORE_STOPWORDS else 1.0

    ta = _tok(a)
    tb = set(_tok(b))
    if not ta: return 0.0
    num = sum(_w(t) for t in ta if t in tb)
    den = sum(_w(t) for t in ta)
    return round(num / den, 4) if den > 0 else 0.0

def extract_claims(text: str) -> list:
    """Split text into atomic claim sentences (> 3 tokens)."""
    if not isinstance(text, str) or not text.strip():
        return []
    return [s for s in sent_tokenize(text.strip()) if len(s.split()) > 3]

# ── E: Evidence Grounding (Eq. 7) ─────────────────────────────────────────────
def compute_E(output: str, evidence_texts=None, threshold=0.25) -> float:
    """
    Fraction of output claims with Jaccard overlap > threshold
    against at least one retrieved evidence sentence.
    Returns 0.0 if no evidence available.
    """
    claims = extract_claims(output)
    if not claims or not evidence_texts:
        return 0.0
    supported = sum(
        1 for c in claims
        if max((jaccard(c, e) for e in evidence_texts), default=0) > threshold
    )
    return supported / len(claims)

# ── V: Verifiability (Eq. 8) ──────────────────────────────────────────────────
def compute_V(label: str) -> float:
    """correct→1.0, partial→0.5 (alpha), incorrect→0.0"""
    return {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}.get(
        label.strip().lower(), 0.5
    )

# ── S: Stability Under Perturbation (Eq. 9) ───────────────────────────────────
def _first_sentence(text: str) -> str:
    """Extract the first substantive sentence from an output."""
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if len(s.split()) > 3]
    return sents[0] if sents else text[:120]

def _sym_containment(a: str, b: str) -> float:
    """Symmetric IDF containment: mean of forward and backward containment."""
    return (jaccard(a, b) + jaccard(b, a)) / 2.0

def compute_S(base_output: str, alt_outputs: list) -> float:
    """
    Stability: do alternative phrasings of the prompt produce consistent facts?

    Two-component approach:
      40% Lexical  — symmetric IDF containment on the first (core) sentence of
                     each output vs the base. First sentence = the direct answer,
                     stripping elaboration that legitimately varies by style.
      60% Semantic — single Groq call asking: are all these outputs factually
                     consistent? 0-10 score, normalised. One call, not 4.

    Why this is better than plain Jaccard on full outputs:
      Jaccard on full outputs penalises length differences and style variation
      even when the underlying facts agree perfectly. A 1-sentence and a
      5-sentence answer to the same question can score Jaccard=0.10 while
      being 100% semantically consistent.
    """
    if not base_output or not alt_outputs:
        return 0.0

    # Lexical component — symmetric containment on core sentences
    base_core = _first_sentence(base_output)
    lex_scores = []
    for alt in alt_outputs:
        alt_core = _first_sentence(alt)
        lex_scores.append(_sym_containment(base_core, alt_core))
    lex_S = float(np.mean(lex_scores))

    # Semantic component — Groq consistency judge (single call)
    try:
        alts_block = "\n".join(f"  Alt {i+1}: {a[:200]}" for i, a in enumerate(alt_outputs))
        system = (
            "You are a factual consistency checker. "
            "Given a base answer and several alternative answers to the same question, "
            "rate 0-10 how consistently they all assert the same core facts:\n"
            "  0  = directly contradictory\n"
            "  5  = partially consistent, some facts differ\n"
            "  10 = all outputs agree on every core fact\n"
            "Reply with ONE integer only."
        )
        prompt = f"Base answer: {base_output[:300]}\n\nAlternatives:\n{alts_block}"
        raw    = call_groq(prompt, system=system, max_tokens=5)
        nums   = re.findall(r'\d+', raw)
        sem_S  = min(int(nums[0]), 10) / 10.0 if nums else 0.5
    except Exception:
        sem_S = lex_S   # fallback to lexical if Groq fails

    return round(0.40 * lex_S + 0.60 * sem_S, 4)

# ── L: Logical & Numerical Coherence (Eq. 10) ─────────────────────────────────
def compute_L(output: str) -> float:
    """
    Rule-based checks — returns fraction of rules NOT violated.
    Rule 1: self-contradiction  (e.g. 'X is and is not Y')
    Rule 2: arithmetic errors   (e.g. '2 + 3 = 6')
    Rule 3: unit mixing         (km and miles in same text)
    """
    violations, total = 0, 3
    if re.search(r'\b\w+\s+is\s+(and\s+is\s+not|but\s+is\s+not)\b', output, re.IGNORECASE):
        violations += 1
    for a, op, b, result in re.findall(
        r'(\d+(?:\.\d+)?)\s*([+\-])\s*(\d+(?:\.\d+)?)\s*=\s*(\d+(?:\.\d+)?)', output
    ):
        expected = float(a) + float(b) if op == '+' else float(a) - float(b)
        if abs(expected - float(result)) > 1e-6:
            violations += 1; break
    if (re.search(r'\b(km|kilometers?)\b', output, re.IGNORECASE) and
            re.search(r'\bmiles?\b', output, re.IGNORECASE)):
        violations += 1
    return 1.0 - violations / total

# ── C: Calibration (Eq. 11) ───────────────────────────────────────────────────
def compute_C(hat_p: float, V: float) -> float:
    """C = 1 − |hat_p − V|   Penalises over- and under-confidence."""
    return 1.0 - abs(hat_p - V)

# ── T: Composite Trust Score (Eq. 12) ─────────────────────────────────────────
def compute_T(E, V, S, L, C, risk_level=0.5, weights=None) -> float:
    """
    Composite trust score T in [0, 100].

    Formula: rescaled sigmoid applied to the weighted mean of signals.

    Why rescaled sigmoid?
      The paper uses T = 100·σ(Σλᵢzᵢ - b(Rₜ)), but signals zᵢ ∈ [0,1].
      With [0,1] inputs, the raw sigmoid only spans σ(-risk) to σ(1-risk),
      e.g. 45–69 at risk=0.2 — so perfect signals never reach 100.
      We rescale so the sigmoid's actual min/max for these inputs always
      maps to 0 and 100, preserving non-linearity and risk sensitivity
      without compressing the range.

    Rescaling:
      u     = dot(λ_norm, z) - risk_level
      T_raw = σ(u)
      T_min = σ(0.0 - risk_level)   [floor: all signals = 0]
      T_max = σ(1.0 - risk_level)   [ceiling: all signals = 1]
      T     = 100 · (T_raw - T_min) / (T_max - T_min)

    Risk effect:
      Higher risk_level shifts u downward, which lowers T — same direction
      as the paper intends (high-risk tasks require stronger evidence to pass).
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS
    signals = [E, V, S, L, C]
    lam     = np.array(weights, dtype=float)
    mask    = np.array([s is not None for s in signals])
    z       = np.array([s if s is not None else 0.0 for s in signals], dtype=float)
    lam_active = lam * mask
    if lam_active.sum() == 0:
        return 50.0
    lam_norm = lam_active / lam_active.sum()

    def sigmoid(x): return 1.0 / (1.0 + math.exp(-x))

    u     = float(np.dot(lam_norm, z) - risk_level)
    T_raw = sigmoid(u)
    T_min = sigmoid(0.0 - risk_level)   # all signals = 0
    T_max = sigmoid(1.0 - risk_level)   # all signals = 1

    if abs(T_max - T_min) < 1e-9:
        return 50.0
    return round(100.0 * (T_raw - T_min) / (T_max - T_min), 2)

def route(score: float):
    if score >= THRESHOLD_PUBLISH: return "✅ PUBLISH", "#27ae60"
    elif score >= THRESHOLD_REVIEW: return "⚠️  REVIEW", "#e67e22"
    else:                           return "🚫 BLOCK",   "#e74c3c"

print("✅ Trust signal functions loaded.")




✅ Trust signal functions loaded.


## Groq API Integration

In [13]:
# ── Cell 4: Groq API Integration ──────────────────────────────────────────────
import time, json
import textwrap
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

# Paraphrase styles designed for SEMANTIC stability testing.
# Goal: same meaning expressed differently — not style variation.
# Style variation (formal/casual/step-by-step) intentionally changes surface form,
# which unfairly penalises the stability score even when facts are consistent.
PARAPHRASE_STYLES = [
    "Rephrase the answer using completely different words but preserve the exact meaning:",
    "Answer as if explaining to someone unfamiliar with the topic, using plain language:",
    "Give the most concise factual answer possible, stripping all elaboration:",
    "Answer from a different angle — explain WHY or HOW rather than just stating the fact:",
]

def call_groq(prompt: str, system: str = "", max_tokens: int = 512,
              retries: int = 3) -> str:
    """
    Send a prompt to Groq and return the text response.
    Automatically retries on 429 rate-limit errors with back-off.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model      = GROQ_MODEL,
                messages   = messages,
                max_tokens = max_tokens,
                temperature= 0.2,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "429" in err and attempt < retries - 1:
                # Parse wait time from error if available
                wait = 10 * (attempt + 1)
                import re as _re
                match = _re.search(r'retry.*?(\d+(?:\.\d+)?)s', err, _re.IGNORECASE)
                if match:
                    wait = float(match.group(1)) + 2
                print(f"   ⚠️  Rate limit hit — waiting {wait:.0f}s before retry {attempt+2}/{retries}…")
                time.sleep(wait)
            else:
                raise

# ── Verifiability ─────────────────────────────────────────────────────────────
def verify_output(prompt: str, output: str) -> str:
    """
    Returns 'correct', 'partial', or 'incorrect'.

    Key design: evaluates FACTUAL ACCURACY of the claims made in the answer,
    not whether the answer is positive or negative.

    A correct 'No' answer to a false premise → 'correct'
    An incorrect 'Yes' answer to a false premise → 'incorrect'

    Without this distinction, negation answers ("No, X is not Y. The correct
    answer is Z.") are misclassified as 'incorrect' because the verifier
    conflates the negative polarity of the response with factual inaccuracy.
    This cascades into C = 1 - |hat_p - V| also collapsing to 0.0.
    """
    system = (
        "You are a factual accuracy verifier. "
        "Evaluate whether the FACTUAL CLAIMS in the answer are accurate. "
        "Do NOT classify based on whether the answer is positive or negative, "
        "and do NOT classify based on whether it confirms the question. "
        "A correct 'No' answer that accurately corrects a false premise is 'correct'. "
        "A 'Yes' answer that confirms a false premise is 'incorrect'. "
        "Classify as: "
        "'correct' (all factual claims in the answer are accurate), "
        "'partial' (some claims accurate, some wrong or missing), "
        "'incorrect' (the answer contains factually wrong information). "
        "Reply with ONE word only: correct, partial, or incorrect."
    )
    raw     = call_groq(f"Question: {prompt}\n\nAnswer: {output}",
                         system=system, max_tokens=10)
    cleaned = re.sub(r'[^a-z]', '', raw.lower())
    return cleaned if cleaned in ("correct", "partial", "incorrect") else "partial"

# ── Stability ─────────────────────────────────────────────────────────────────
def get_paraphrased_outputs(prompt: str) -> list:
    """4 paraphrase-style answers used to compute S."""
    outputs = []
    for style in PARAPHRASE_STYLES:
        outputs.append(call_groq(f"{style}\n\n{prompt}", max_tokens=256))
        time.sleep(0.5)   # small gap to respect rate limits
    return outputs

# ── Calibration ───────────────────────────────────────────────────────────────
def get_confidence(prompt: str, output: str) -> float:
    """Self-reported confidence 0–100, normalised to [0,1]."""
    system = (
        "You are an uncertainty-aware AI. "
        "Given a question and your answer, reply with ONE integer (0–100) "
        "for how confident you are the answer is factually correct. "
        "0 = no confidence, 100 = completely certain. Integer only, no other text."
    )
    raw  = call_groq(f"Question: {prompt}\n\nAnswer: {output}",
                      system=system, max_tokens=10)
    nums = re.findall(r'\d+', raw)
    return min(int(nums[0]), 100) / 100.0 if nums else 0.5

print("✅ Groq API functions loaded.")
print(f"   Model in use : {GROQ_MODEL}")

# Quick connectivity test
try:
    _test = call_groq("Reply with the single word: hello", max_tokens=10)
    print(f"   API test     : ✅ connected  (response: '{_test[:30]}')")
except Exception as e:
    print(f"   API test     : ❌ FAILED — {e}")
    print("   → Double-check your GROQ_API_KEY in Cell 2 and re-run that cell.")



✅ Groq API functions loaded.
   Model in use : llama-3.3-70b-versatile
   API test     : ❌ FAILED — Connection error.
   → Double-check your GROQ_API_KEY in Cell 2 and re-run that cell.


*   **E**: "Is the output backed by real, retrievable sources?"
*   **V**: "Does the output agree with known facts?"
*   **S**: "Does rephrasing the question change the answer? A trustworthy output should reach the same conclusion regardless of how the question is worded."
*   **L**: "Are there internal contradictions, arithmetic errors, or mixed units?"
*   **C**: "Is the model's expressed confidence appropriate? Overconfident outputs on uncertain facts are penalised."

## Multi-Source Evidence Grounding

In [14]:
# ── Cell 4b: Multi-Source Evidence Grounding ──────────────────────────────────
# Sources: Wikipedia + DuckDuckGo Instant Answer API (both free, no key needed)
# Each claim in the AI output is scored against both sources independently,
# then combined via confidence-weighted voting.
import requests, urllib.parse

WIKI_MW      = "https://en.wikipedia.org/w/api.php"
WIKI_REST    = "https://en.wikipedia.org/api/rest_v1/page/summary"
DDG_API      = "https://api.duckduckgo.com/"
WIKI_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; TrustScoreDashboard/1.0)"}
DDG_HEADERS  = {"User-Agent": "Mozilla/5.0 (compatible; TrustScoreDashboard/1.0)"}

AGREE_THRESHOLD = 0.25   # sources are "contested" if they differ by more than this

# ══════════════════════════════════════════════════════════════════════════════
# WIKIPEDIA RETRIEVAL
# ══════════════════════════════════════════════════════════════════════════════

def extract_search_keywords(prompt):
    system = (
        "You are a Wikipedia search assistant. "
        "Convert the question into 2-4 keywords matching a Wikipedia article title. "
        "Output ONLY keywords, lowercase, no punctuation. "
        "Q: What is the boiling point of water at sea level? -> boiling point water"
    )
    try:
        kw = call_groq(prompt, system=system, max_tokens=15)
        kw = re.sub(r"[^\w\s]", "", kw).strip().lower()
        return kw if kw else _kw_fallback(prompt)
    except Exception:
        return _kw_fallback(prompt)

def _kw_fallback(prompt):
    stop = {"what","is","the","a","an","how","why","who","when","where",
            "does","do","did","are","was","were","will","can","could",
            "at","of","in","on","to","for","with","by","from"}
    words = [w for w in re.sub(r"[^\w\s]","",prompt.lower()).split() if w not in stop]
    return " ".join(words[:4])

def wiki_get_candidates(keywords, n=5):
    params = {"action":"query","list":"search","srsearch":keywords,
              "srlimit":n,"format":"json","srnamespace":0}
    try:
        r = requests.get(WIKI_MW, params=params, headers=WIKI_HEADERS, timeout=10)
        r.raise_for_status()
        return [h["title"] for h in r.json().get("query",{}).get("search",[])]
    except requests.exceptions.ConnectionError:
        print("   \u274c Wikipedia unreachable"); return []
    except Exception as e:
        print(f"   \u274c Wiki search error: {e}"); return []

def wiki_select_best(prompt, candidates):
    if not candidates: return None
    if len(candidates) == 1: return candidates[0]
    numbered = "\n".join(f"{i+1}. {t}" for i,t in enumerate(candidates))
    system = (
        "You are a Wikipedia article selector. "
        "Reply with ONLY the number of the article most likely to directly answer the question. "
        "Single integer only."
    )
    try:
        raw  = call_groq(f"Question: {prompt}\n\nArticles:\n{numbered}", system=system, max_tokens=5)
        nums = re.findall(r"\d+", raw)
        idx  = max(0, min(int(nums[0])-1, len(candidates)-1)) if nums else 0
        return candidates[idx]
    except Exception:
        return candidates[0]

def wiki_fetch_article(title):
    """
    Fetch Wikipedia article text.
    Primary  : MediaWiki extracts API — returns up to 30 sentences, most reliable.
    Fallback : REST summary — shorter but faster.
    """
    EMPTY   = {"title":title,"summary":"","sentences":[],"url":""}
    encoded = urllib.parse.quote(title.replace(" ","_"), safe="")
    page_url = f"https://en.wikipedia.org/wiki/{encoded}"

    # Primary: MediaWiki extracts (richer, more sentences)
    try:
        params = {"action":"query","format":"json","titles":title,"prop":"extracts",
                  "exintro":True,"explaintext":True,"exsectionformat":"plain",
                  "exsentences":30}
        r = requests.get(WIKI_MW, params=params, headers=WIKI_HEADERS, timeout=10)
        r.raise_for_status()
        pages   = r.json().get("query",{}).get("pages",{})
        page    = next(iter(pages.values()),{})
        summary = page.get("extract","").strip()
        if summary and len(summary.split()) > 20:
            sents = [s.strip() for s in sent_tokenize(summary) if len(s.split())>4]
            return {"title":page.get("title",title),"summary":summary,
                    "sentences":sents,"url":page_url}
    except Exception:
        pass

    # Fallback: REST summary
    try:
        r = requests.get(f"{WIKI_REST}/{encoded}", headers=WIKI_HEADERS, timeout=10)
        if r.status_code == 200:
            data    = r.json()
            summary = data.get("extract","").strip()
            if summary:
                sents = [s.strip() for s in sent_tokenize(summary) if len(s.split())>4]
                return {"title":data.get("title",title),"summary":summary,"sentences":sents,
                        "url":data.get("content_urls",{}).get("desktop",{}).get("page",page_url)}
    except Exception as e:
        print(f"   \u274c Wiki fetch failed: {e}")

    return EMPTY

def fetch_wikipedia_source(prompt, vlog):
    vlog("  [Wikipedia] Extracting keywords...")
    kw = extract_search_keywords(prompt)
    vlog(f"  [Wikipedia] Keywords: \"{kw}\"")
    time.sleep(0.5)
    candidates = wiki_get_candidates(kw, n=5)
    if not candidates:
        return {"sentences":[],"url":None,"title":None,"confidence":0.0,"error":"no_candidates"}
    vlog(f"  [Wikipedia] Candidates: {', '.join(repr(c) for c in candidates)}")
    best = wiki_select_best(prompt, candidates)
    vlog(f"  [Wikipedia] \U0001f916 Selected: \"{best}\"")
    time.sleep(0.5)
    article = wiki_fetch_article(best)
    sents   = article["sentences"]
    conf    = min(1.0, len(sents) / 10.0)
    vlog(f"  [Wikipedia] \u2705 {len(sents)} sentences (conf={conf:.2f}) | {article['url']}")
    if sents: vlog(f"  [Wikipedia] Preview: \"{sents[0][:70]}...\"")
    return {"sentences":sents,"url":article["url"],"title":article["title"],
            "confidence":conf,"error":None}

# ══════════════════════════════════════════════════════════════════════════════
# DUCKDUCKGO RETRIEVAL
# ══════════════════════════════════════════════════════════════════════════════

def fetch_ddg_source(prompt, vlog):
    vlog("  [DuckDuckGo] Querying Instant Answer API...")
    params = {"q":prompt,"format":"json","no_html":"1","skip_disambig":"1"}
    EMPTY  = {"sentences":[],"url":None,"title":None,"confidence":0.0,"error":None}
    try:
        r    = requests.get(DDG_API, params=params, headers=DDG_HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
    except requests.exceptions.ConnectionError:
        vlog("  [DuckDuckGo] \u274c Unreachable")
        return {**EMPTY, "error":"unreachable"}
    except Exception as e:
        vlog(f"  [DuckDuckGo] \u274c Error: {e}")
        return {**EMPTY, "error":str(e)}

    sentences = []
    for field in ["Abstract","Answer","Definition"]:
        text = data.get(field,"").strip()
        if text and len(text.split()) > 2:
            sentences += [s.strip() for s in sent_tokenize(text) if len(s.split())>4]
    for topic in data.get("RelatedTopics",[])[:5]:
        text = topic.get("Text","").strip() if isinstance(topic,dict) else ""
        if text and len(text.split()) > 5:
            sentences.append(text[:200])
    seen, unique = set(), []
    for s in sentences:
        if s not in seen: seen.add(s); unique.append(s)
    sentences = unique

    src_url   = data.get("AbstractURL","") or data.get("DefinitionURL","")
    src_title = data.get("Heading","") or "DuckDuckGo"
    conf      = min(1.0, len(sentences) / 5.0)

    if sentences:
        vlog(f"  [DuckDuckGo] \u2705 {len(sentences)} sentences (conf={conf:.2f})")
        vlog(f"  [DuckDuckGo] Preview: \"{sentences[0][:70]}...\"")
        if src_url: vlog(f"  [DuckDuckGo] \U0001f517 {src_url}")
    else:
        vlog("  [DuckDuckGo] \u26a0\ufe0f  No instant answer found for this query")
    return {"sentences":sentences,"url":src_url,"title":src_title,"confidence":conf,"error":None}

# ══════════════════════════════════════════════════════════════════════════════
# CLAIM SCORER + VOTER
# ══════════════════════════════════════════════════════════════════════════════

def _groq_entailment(claim, evidence):
    system = (
        "You are an expert fact-checker. "
        "Rate how strongly the EVIDENCE supports the CLAIM on a 0-10 scale: "
        "0=contradicts/unrelated, 5=partially relevant, 10=fully supports. "
        "Reply with ONE integer only."
    )
    try:
        raw  = call_groq(f"Evidence: {evidence}\nClaim: {claim}", system=system, max_tokens=5)
        nums = re.findall(r"\d+", raw)
        return min(int(nums[0]),10)/10.0 if nums else 0.5
    except Exception:
        return 0.5

def score_claim_vs_source(claim, sentences):
    """
    Score one claim against a list of evidence sentences.

    Algorithm:
      1. Score claim vs every sentence using IDF-weighted containment
         (fraction of claim content-words found in the evidence sentence)
      2. Take the best-matching sentence
      3. Ask Groq to rate entailment (0-10) on that sentence
      4. Final = 0.60 * containment + 0.40 * groq_entailment

    Why containment not Jaccard:
      Jaccard divides by the UNION, so long evidence sentences score low
      even when the claim is fully contained within them.
      Containment divides by the CLAIM size — the right denominator for
      the question "is this claim supported by the evidence?"
    """
    if not sentences: return 0.0
    # jaccard() is now IDF-weighted containment — best score across all sentences
    lex_scores = [jaccard(claim, s) for s in sentences]
    best_lex   = max(lex_scores)
    best_sent  = sentences[lex_scores.index(best_lex)]
    sem        = _groq_entailment(claim, best_sent)
    return round(0.60 * best_lex + 0.40 * sem, 4)

def vote_claim_scores(scores_by_source, confidences):
    active = {src:sc for src,sc in scores_by_source.items() if sc is not None}
    if not active: return {"score":0.0,"contested":False,"sources_used":[]}
    total_w = sum(confidences.get(s,0.5) for s in active)
    voted   = sum(sc*confidences.get(s,0.5) for s,sc in active.items()) / max(total_w,1e-9)
    vals    = list(active.values())
    contested = (len(vals)>=2 and
                 max(abs(vals[i]-vals[j]) for i in range(len(vals)) for j in range(i+1,len(vals)))
                 > AGREE_THRESHOLD)
    return {"score":round(voted,4),"contested":contested,"sources_used":list(active.keys())}

# ══════════════════════════════════════════════════════════════════════════════
# MAIN E PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def compute_E_multisource(prompt, output, verbose=True):
    log = lambda msg: print(f"      {msg}") if verbose else None

    log("\U0001f310 Retrieving from Wikipedia...")
    wiki_src = fetch_wikipedia_source(prompt, log)
    time.sleep(1)

    log("\U0001f986 Retrieving from DuckDuckGo...")
    ddg_src  = fetch_ddg_source(prompt, log)
    time.sleep(0.5)

    sources     = {"wikipedia":wiki_src,"duckduckgo":ddg_src}
    confidences = {n:s["confidence"] for n,s in sources.items()}

    if sum(len(s["sentences"]) for s in sources.values()) == 0:
        log("\u26a0\ufe0f  Both sources returned no content - E = 0.0")
        return 0.0, {"sources":sources,"claim_scores":[],"contested_count":0,
                     "E_per_source":{},"error":"no_content"}

    log(f"\U0001f4ca Evidence: Wiki={len(wiki_src['sentences'])} sents, DDG={len(ddg_src['sentences'])} sents")

    claims = extract_claims(output)
    if not claims:
        log("\u26a0\ufe0f  No claims extracted - E = 0.0")
        return 0.0, {"sources":sources,"claim_scores":[],"contested_count":0,
                     "E_per_source":{},"error":"no_claims"}

    log(f"\U0001f4dd Scoring {len(claims)} claim(s) across both sources...")
    claim_scores = []
    for i, claim in enumerate(claims):
        wiki_sc = score_claim_vs_source(claim, wiki_src["sentences"])
        ddg_sc  = score_claim_vs_source(claim, ddg_src["sentences"])
        time.sleep(0.3)
        voted = vote_claim_scores(
            {"wikipedia": wiki_sc if wiki_src["sentences"] else None,
             "duckduckgo": ddg_sc  if ddg_src["sentences"]  else None},
            confidences
        )
        short     = claim[:60] + ("..." if len(claim)>60 else "")
        bar       = "\u2588" * int(voted["score"]*20) + "\u2591" * (20-int(voted["score"]*20))
        flag      = " \u26a0\ufe0f CONTESTED" if voted["contested"] else ""
        log(f"   [{bar}] {voted['score']:.3f}  W={wiki_sc:.2f} D={ddg_sc:.2f}  \"{short}\"{flag}")
        claim_scores.append({"claim":claim,"wiki_score":wiki_sc,"ddg_score":ddg_sc,
                              "voted_score":voted["score"],"contested":voted["contested"]})

    E_score = float(np.mean([cs["voted_score"] for cs in claim_scores]))
    contested_n = sum(1 for cs in claim_scores if cs["contested"])
    E_per = {
        "wikipedia":  round(float(np.mean([cs["wiki_score"] for cs in claim_scores])),4),
        "duckduckgo": round(float(np.mean([cs["ddg_score"]  for cs in claim_scores])),4),
    }
    log(f"\u2705 E = {E_score:.3f}  (Wiki={E_per['wikipedia']:.3f}, DDG={E_per['duckduckgo']:.3f}, contested={contested_n}/{len(claims)})")
    return E_score, {"sources":sources,"claim_scores":claim_scores,
                     "contested_count":contested_n,"E_per_source":E_per,"error":None}

# Backward-compat alias so evaluate_trust() works unchanged
compute_E_wikipedia = compute_E_multisource

# ══════════════════════════════════════════════════════════════════════════════
# CONNECTIVITY + SMOKE TEST
# ══════════════════════════════════════════════════════════════════════════════
print("\u2705 Multi-Source Evidence Grounding loaded.")
print("   Sources : Wikipedia + DuckDuckGo Instant Answer (both free, no key)")
print("   Scoring : 60% Jaccard + 40% Groq entailment per claim per source")
print("   Voting  : confidence-weighted mean  |  contested flag when sources disagree")
print()
print("\U0001f50c Testing sources...")
try:
    _rw = requests.get(WIKI_MW, params={"action":"query","format":"json","titles":"Water","prop":"info"},
                       headers=WIKI_HEADERS, timeout=6)
    print("   \u2705 Wikipedia  :", "reachable" if "query" in _rw.json() else "unexpected")
except Exception as _e:
    print(f"   \u274c Wikipedia  : {type(_e).__name__}")
try:
    _rd = requests.get(DDG_API, params={"q":"boiling point water","format":"json","no_html":"1"},
                       headers=DDG_HEADERS, timeout=6)
    _ab = _rd.json().get("Abstract","")[:65] or _rd.json().get("Answer","")[:65]
    print(f"   \u2705 DuckDuckGo : reachable  |  Abstract: \"{_ab}...\"" if _ab
          else "   \u26a0\ufe0f  DuckDuckGo : reachable but no abstract for test query")
except Exception as _e:
    print(f"   \u274c DuckDuckGo : {type(_e).__name__}")

print()
print("\U0001f50e Smoke test (Wikipedia selection)...")
_kw    = extract_search_keywords("What is the boiling point of water at sea level?")
_cands = wiki_get_candidates(_kw, n=5)
_best  = wiki_select_best("What is the boiling point of water at sea level?", _cands) if _cands else None
print(f"   Keywords   : \"{_kw}\"")
print(f"   Candidates : {_cands[:4]}")
print(f"   \U0001f916 Selected  : \"{_best}\"")
if _best:
    _art = wiki_fetch_article(_best)
    if _art["sentences"]:
        print(f"   \u2705 Fetched {len(_art['sentences'])} sentences from \"{_art['title']}\": \"{_art['sentences'][0][:70]}...\"")





✅ Multi-Source Evidence Grounding loaded.
   Sources : Wikipedia + DuckDuckGo Instant Answer (both free, no key)
   Scoring : 60% Jaccard + 40% Groq entailment per claim per source
   Voting  : confidence-weighted mean  |  contested flag when sources disagree

🔌 Testing sources...
   ✅ Wikipedia  : reachable
   ⚠️  DuckDuckGo : reachable but no abstract for test query

🔎 Smoke test (Wikipedia selection)...
   Keywords   : "boiling point water sea"
   Candidates : ['Boiling', 'Boiling water reactor', 'Boiling point', 'Atmospheric pressure']
   🤖 Selected  : "Boiling"
   ✅ Fetched 10 sentences from "Boiling": "Boiling or ebullition is the rapid phase transition from liquid to gas..."


In [15]:
# ── Cell 5: Visualisation Helpers ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

SIGNAL_LABELS = ['E\nGrounding', 'V\nVerifiability', 'S\nStability', 'L\nCoherence', 'C\nCalibration']
SIGNAL_COLORS = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#1abc9c']

def plot_trust_dashboard(prompt, output, E, V, S, L, C,
                          trust_score, risk_level, weights,
                          verdict, routing_label, routing_color,
                          wiki_title=None, wiki_url=None):
    signals = [E, V, S, L, C]
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0f1117')
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.35)

    # ── 1. Gauge ──────────────────────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.set_facecolor('#1a1d26')
    theta = np.linspace(np.pi, 0, 300)
    ax0.plot(np.cos(theta), np.sin(theta), color='#2c2f3a', linewidth=18, solid_capstyle='round')
    frac = trust_score / 100.0
    tf   = np.linspace(np.pi, np.pi - frac * np.pi, 300)
    ax0.plot(np.cos(tf), np.sin(tf), color=routing_color, linewidth=18, solid_capstyle='round')
    ax0.text(0, -0.15, f"{trust_score:.1f}",  ha='center', va='center',
             fontsize=32, fontweight='bold', color='white')
    ax0.text(0, -0.45, "Trust Score",          ha='center', va='center', fontsize=11, color='#aaa')
    ax0.text(0, -0.70, routing_label,           ha='center', va='center',
             fontsize=13, fontweight='bold', color=routing_color)
    ax0.set_xlim(-1.3, 1.3); ax0.set_ylim(-0.9, 1.15); ax0.axis('off')
    ax0.set_title('Composite Trust Score', color='white', fontsize=12, pad=6)

    # ── 2. Horizontal bars ────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.set_facecolor('#1a1d26')
    bars = ax1.barh(SIGNAL_LABELS, signals, color=SIGNAL_COLORS, edgecolor='none', height=0.55)
    for bar, val in zip(bars, signals):
        ax1.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                 f'{val:.2f}', va='center', ha='left', color='white', fontsize=10)
    ax1.set_xlim(0, 1.2)
    ax1.axvline(0.5, color='#444', linestyle='--', linewidth=0.8)
    ax1.tick_params(axis='y', colors='white', labelsize=9)
    ax1.tick_params(axis='x', colors='#aaa')
    ax1.spines[:].set_visible(False)
    ax1.set_title('Trust Signal Breakdown', color='white', fontsize=12, pad=6)

    # ── 3. Radar chart ────────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 2], polar=True)
    ax2.set_facecolor('#1a1d26')
    N      = 5
    angles = [n / N * 2 * np.pi for n in range(N)] + [0]
    vals   = signals + signals[:1]
    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(['E', 'V', 'S', 'L', 'C'], color='white', size=11)
    ax2.set_ylim(0, 1)
    ax2.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax2.set_yticklabels(['0.25','0.5','0.75','1.0'], color='#666', size=7)
    ax2.grid(color='#2c2f3a', linewidth=0.8)
    ax2.spines['polar'].set_color('#2c2f3a')
    ax2.plot(angles, vals, color='#3498db', linewidth=2)
    ax2.fill(angles, vals, alpha=0.25, color='#3498db')
    ax2.set_title('Signal Radar', color='white', fontsize=12, pad=15)

    # ── 4. Weighted contributions ─────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_facecolor('#1a1d26')
    wts    = np.array(weights, dtype=float); wts /= wts.sum()
    contrib = np.array(signals) * wts
    ax3.bar(['E','V','S','L','C'], contrib, color=SIGNAL_COLORS, edgecolor='none')
    for i, (c, w) in enumerate(zip(contrib, wts)):
        ax3.text(i, c + 0.004, f'λ={w:.0%}', ha='center', va='bottom', color='white', fontsize=8)
    ax3.set_ylabel('Weighted Contribution', color='#aaa')
    ax3.tick_params(colors='white'); ax3.spines[:].set_visible(False)
    ax3.set_title('Weighted Signal Contributions', color='white', fontsize=12, pad=6)

    # ── 5. Risk dial ──────────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_facecolor('#1a1d26')
    tbg = np.linspace(np.pi, 0, 300)
    ax4.plot(np.cos(tbg), np.sin(tbg), color='#2c2f3a', linewidth=20, solid_capstyle='round')
    for t0, t1, col in [(np.pi, np.pi*2/3,'#27ae60'),
                         (np.pi*2/3, np.pi/3,'#f1c40f'),
                         (np.pi/3, 0,'#e74c3c')]:
        t = np.linspace(t0, t1, 100)
        ax4.plot(np.cos(t), np.sin(t), color=col, linewidth=20, solid_capstyle='butt')
    needle = np.pi * (1 - risk_level)
    ax4.annotate('', xy=(0.75*np.cos(needle), 0.75*np.sin(needle)), xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='white', lw=2.5))
    ax4.text(0, -0.25, f'Risk: {risk_level:.1f}', ha='center', va='center',
             fontsize=14, fontweight='bold', color='white')
    ax4.text(-1.0, -0.05, 'Low', color='#27ae60', fontsize=8)
    ax4.text( 0.75,-0.05, 'High',color='#e74c3c', fontsize=8)
    ax4.set_xlim(-1.3,1.3); ax4.set_ylim(-0.5,1.15); ax4.axis('off')
    ax4.set_title('Task Risk Level', color='white', fontsize=12, pad=6)

    # ── 6. Summary panel ─────────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.set_facecolor('#1a1d26'); ax5.axis('off')
    items = [
        ("📝 Prompt",   textwrap.fill(prompt[:115] + ('…' if len(prompt)>115 else ''), 36)),
        ("🤖 Output",   textwrap.fill(output[:115] + ('…' if len(output)>115 else ''), 36)),
        ("🔎 Verdict",  verdict.capitalize()),
        ("📊 Score",    f"{trust_score:.1f} / 100"),
        ("🚦 Decision", routing_label),
        ("🤖 Model",    GROQ_MODEL),
        ("📚 Wiki Src", wiki_title or "N/A"),
    ]
    y = 0.97
    for label, value in items:
        ax5.text(0.02, y,      label, transform=ax5.transAxes,
                 color='#aaa', fontsize=9, va='top', fontweight='bold')
        ax5.text(0.02, y-0.06, value, transform=ax5.transAxes,
                 color='white', fontsize=8, va='top', multialignment='left')
        y -= 0.17
    ax5.set_title('Summary', color='white', fontsize=12, pad=6)

    fig.suptitle('🔍 AI Trust Score Dashboard  •  Powered by Groq + LLaMA 3.3',
                 color='white', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

print("✅ Visualisation helpers loaded.")

✅ Visualisation helpers loaded.


In [16]:
# ── Cell 6: Full Evaluation Pipeline ──────────────────────────────────────────
def evaluate_trust(
    prompt:        str,
    pasted_output: str   = None,
    risk_level:    float = None,
    weights:       list  = None,
    verbose:       bool  = True,
) -> dict:
    """
    Full multi-signal trust evaluation using Groq + multi-source evidence (Wikipedia + DDG).
    """
    if risk_level is None: risk_level = DEFAULT_RISK_LEVEL
    if weights    is None: weights    = DEFAULT_WEIGHTS

    log = lambda msg: print(f"  → {msg}") if verbose else None
    print(f"\n{'='*62}")
    print( "  TRUST EVALUATION  •  Groq / LLaMA 3.3  +  Wikipedia & DDG")
    print(f"{'='*62}")

    # Step 0 — generate or accept output
    if pasted_output:
        output = pasted_output
        log("Using provided AI output.")
    else:
        log("Generating model response…")
        output = call_groq(prompt)

    _p = prompt[:80] + ('…' if len(prompt)>80 else '')
    _o = output[:80] + ('…' if len(output)>80 else '')
    print(f"\n  📝 Prompt : {_p}")
    print(f"  🤖 Output : {_o}")

    # Step 1 — E: Multi-source evidence grounding
    log("[1/5] Running multi-source evidence grounding…")
    E_val, evidence_meta = compute_E_multisource(prompt, output, verbose=verbose)

    # Extract display info from new meta structure
    wiki_info = evidence_meta.get("sources", {}).get("wikipedia", {})
    ddg_info  = evidence_meta.get("sources", {}).get("duckduckgo", {})
    wiki_title = wiki_info.get("title")
    wiki_url   = wiki_info.get("url")
    ddg_url    = ddg_info.get("url")
    contested_n = evidence_meta.get("contested_count", 0)
    e_per = evidence_meta.get("E_per_source", {})

    if wiki_title:
        log(f"      📚 Wikipedia: {wiki_title}")
    if ddg_url:
        log(f"      🦆 DDG: {ddg_url}")
    time.sleep(1)

    # Step 2 — V: Verifiability
    log("[2/5] Verifying output correctness…")
    verdict = verify_output(prompt, output)
    V_val   = compute_V(verdict)
    log(f"V = {V_val:.3f}  (verdict: {verdict})")
    time.sleep(1)

    # Step 3 — S: Stability
    log("[3/5] Running 4 paraphrase probes for stability…")
    alt_outputs = get_paraphrased_outputs(prompt)
    S_val = compute_S(output, alt_outputs)
    log(f"S = {S_val:.3f}")
    time.sleep(1)

    # Step 4 — L: Logical Coherence
    log("[4/5] Running rule-based coherence checks…")
    L_val = compute_L(output)
    log(f"L = {L_val:.3f}")

    # Step 5 — C: Calibration
    log("[5/5] Requesting self-reported confidence…")
    hat_p = get_confidence(prompt, output)
    C_val = compute_C(hat_p, V_val)
    log(f"C = {C_val:.3f}  (hat_p = {hat_p:.2f})")

    # Composite score
    trust_score = compute_T(E_val, V_val, S_val, L_val, C_val,
                             risk_level=risk_level, weights=weights)
    routing_label, routing_color = route(trust_score)

    print(f"\n  {'─'*54}")
    print(f"  TRUST SCORE : {trust_score:.1f} / 100")
    print(f"  DECISION    : {routing_label}")
    if wiki_title:
        print(f"  WIKI SOURCE : {wiki_title}")
    if e_per:
        print(f"  E breakdown : Wiki={e_per.get('wikipedia',0):.3f}  DDG={e_per.get('duckduckgo',0):.3f}  contested={contested_n}")
    print(f"  {'─'*54}\n")

    # Per-claim breakdown
    claim_scores = evidence_meta.get("claim_scores", [])
    if verbose and claim_scores:
        print("  📋 Per-claim evidence scores  (W=Wikipedia  D=DuckDuckGo):")
        for cs in claim_scores:
            score = cs.get("voted_score", cs.get("score", 0))
            bar   = '█' * int(score * 20) + '░' * (20 - int(score * 20))
            short = cs["claim"][:70] + ('…' if len(cs["claim"])>70 else '')
            flag  = "  ⚠️ CONTESTED" if cs.get("contested") else ""
            wiki_s = cs.get("wiki_score", 0)
            ddg_s  = cs.get("ddg_score", 0)
            print(f"  [{bar}] {score:.2f}  W={wiki_s:.2f} D={ddg_s:.2f}  {short}{flag}")
        print()

    plot_trust_dashboard(
        prompt, output, E_val, V_val, S_val, L_val, C_val,
        trust_score, risk_level, weights,
        verdict, routing_label, routing_color,
        wiki_title=wiki_title,
        wiki_url=wiki_url,
    )

    return {
        "prompt": prompt, "output": output,
        "evidence_meta": evidence_meta,
        "E": E_val, "V": V_val, "S": S_val, "L": L_val, "C": C_val,
        "hat_p": hat_p, "verdict": verdict,
        "trust_score": trust_score, "routing": routing_label,
    }

print("✅ Evaluation pipeline ready.")



✅ Evaluation pipeline ready.


In [17]:
# ── Cell 7: Interactive Widget UI ─────────────────────────────────────────────
#import ipywidgets as widgets
#from IPython.display import display, clear_output

# ── Input fields ──────────────────────────────────────────────────────────────
#prompt_box = widgets.Textarea(
#    value='Is Paris the capital of France?',
#    placeholder='Enter the question or task given to the AI model…',
#    description='Prompt:',
#    layout=widgets.Layout(width='98%', height='80px'),
#    style={'description_width': '65px'}
#)
#paste_box = widgets.Textarea(
#    value='',
#    placeholder=(
#        'Optional: paste the AI-generated response you want to evaluate. '
#        'Leave blank to let the model generate a response automatically.'
#    ),
#    description='AI Output:',
#    layout=widgets.Layout(width='98%', height='100px'),
#    style={'description_width': '70px'}
#)

# ── Risk slider ───────────────────────────────────────────────────────────────
#risk_slider = widgets.FloatSlider(
#    value=DEFAULT_RISK_LEVEL, min=0.0, max=1.0, step=0.05,
#    description='Task Risk:',
#    style={'description_width': '90px'},
#    layout=widgets.Layout(width='62%'),
#    readout_format='.2f', continuous_update=False
#)
#risk_note = widgets.HTML(
#    "<span style='color:#aaa;font-size:12px'>"
#    "0.0 = general / low-stakes &nbsp;·&nbsp; "
#    "0.5 = professional / advisory &nbsp;·&nbsp; "
#    "1.0 = medical / legal / high-stakes"
#    "</span>"
#)

# ── Signal weight sliders ─────────────────────────────────────────────────────
# Each weight (λ) controls how much that signal contributes to the composite score.
# Weights are automatically renormalised to sum to 1.0 inside compute_T().
#_sw = {'description_width': '185px'}
#_sl = widgets.Layout(width='62%')

#w_E = widgets.FloatSlider(
#    value=0.20, min=0, max=1, step=0.05,
#    description='λ E  Evidence Grounding:',
#    style=_sw, layout=_sl,
#    readout_format='.2f', continuous_update=False
#)
#w_V = widgets.FloatSlider(
#    value=0.20, min=0, max=1, step=0.05,
#    description='λ V  Verifiability:',
#    style=_sw, layout=_sl,
#    readout_format='.2f', continuous_update=False
#)
#w_S = widgets.FloatSlider(
#    value=0.20, min=0, max=1, step=0.05,
#    description='λ S  Stability:',
#    style=_sw, layout=_sl,
#    readout_format='.2f', continuous_update=False
#)
#w_L = widgets.FloatSlider(
#    value=0.20, min=0, max=1, step=0.05,
#    description='λ L  Logical Coherence:',
#    style=_sw, layout=_sl,
#    readout_format='.2f', continuous_update=False
#)
#w_C = widgets.FloatSlider(
#    value=0.20, min=0, max=1, step=0.05,
#    description='λ C  Calibration:',
#    style=_sw, layout=_sl,
#    readout_format='.2f', continuous_update=False
#)

# ── Signal legend — plain language explanation for each metric ────────────────
#signal_legend = widgets.HTML("""
#<div style='font-family:sans-serif;font-size:12px;color:#aaa;
#            background:#1a1d26;border-radius:6px;padding:10px 14px;margin-top:4px'>
#  <b style='color:#ccc'>What each signal measures:</b><br><br>
#  <b style='color:#3498db'>E &nbsp;Evidence Grounding</b>
#    &nbsp;—&nbsp; Is the output backed by real, retrievable sources?
#    Claims are checked against Wikipedia and DuckDuckGo.<br><br>
#  <b style='color:#2ecc71'>V &nbsp;Verifiability</b>
#    &nbsp;—&nbsp; Does the output agree with known facts?
#    An independent model verdict: correct / partial / incorrect.<br><br>
#  <b style='color:#9b59b6'>S &nbsp;Stability</b>
#    &nbsp;—&nbsp; Does rephrasing the question change the answer?
#    A trustworthy output should reach the same conclusion regardless of how the question is worded.<br><br>
#  <b style='color:#e67e22'>L &nbsp;Logical Coherence</b>
#    &nbsp;—&nbsp; Are there internal contradictions, arithmetic errors, or mixed units?
#    Checks the output's internal consistency without reference to external facts.<br><br>
#  <b style='color:#1abc9c'>C &nbsp;Calibration</b>
#    &nbsp;—&nbsp; Is the model's expressed confidence appropriate?
#    Overconfident outputs on uncertain facts are penalised.
#</div>
#""")

# ── Routing legend ────────────────────────────────────────────────────────────
#routing_legend = widgets.HTML("""
#<div style='font-family:sans-serif;font-size:12px;color:#aaa;
#            background:#1a1d26;border-radius:6px;padding:10px 14px;margin-top:4px'>
#  <b style='color:#ccc'>Score interpretation &amp; routing:</b><br><br>
#  <b style='color:#27ae60'>✅ PUBLISH &nbsp;(≥ 70)</b>
#    &nbsp;—&nbsp; Output meets the trust threshold. Safe to use or share.<br><br>
#  <b style='color:#e67e22'>⚠️  REVIEW &nbsp;(45 – 69)</b>
#    &nbsp;—&nbsp; Output has weaknesses in one or more signals. Human review recommended before use.<br><br>
#  <b style='color:#e74c3c'>🚫 BLOCK &nbsp;(&lt; 45)</b>
#    &nbsp;—&nbsp; Output fails minimum trust requirements. Do not use without substantial revision.
#</div>
#""")

# ── Run button and output ─────────────────────────────────────────────────────
#run_btn    = widgets.Button(
#    description='▶  Evaluate Trust Score',
#    button_style='primary',
#    layout=widgets.Layout(width='240px', height='42px')
#)
#status_lbl = widgets.HTML(value="")
#out        = widgets.Output()

#def on_run(_):
#    status_lbl.value = "<span style='color:#f1c40f'>⏳ Evaluating — approx. 30–60 seconds…</span>"
#    run_btn.disabled = True
#    with out:
#        clear_output(wait=True)
#        user_prompt = prompt_box.value.strip()
#        user_output = paste_box.value.strip() or None
#        if not user_prompt:
#            print("⚠️  Please enter a prompt before running.")
#            status_lbl.value = ""; run_btn.disabled = False; return
#        try:
#            evaluate_trust(
#                prompt        = user_prompt,
#                pasted_output = user_output,
#                risk_level    = risk_slider.value,
#                weights       = [w_E.value, w_V.value, w_S.value, w_L.value, w_C.value],
#            )
#            status_lbl.value = "<span style='color:#2ecc71'>✅ Evaluation complete.</span>"
#        except Exception as e:
#            print(f"\n❌ Error: {e}")
#            print("→ Check your GROQ_API_KEY in Cell 2 and re-run that cell.")
#            status_lbl.value = "<span style='color:#e74c3c'>❌ Error — see output above.</span>"
#    run_btn.disabled = False

#run_btn.on_click(on_run)

# ── Layout helpers ────────────────────────────────────────────────────────────
#def H(text):
#    return widgets.HTML(f"<b style='font-family:sans-serif;font-size:13px;color:#ddd'>{text}</b>")

#def divider():
#    return widgets.HTML("<hr style='border:0;border-top:1px solid #2c2f3a;margin:8px 0'>")

#header = widgets.HTML("""
#<div style='font-family:sans-serif;padding-bottom:4px'>
#  <h2 style='margin-bottom:2px;color:#fff'>🔍 AI Trust Score Dashboard</h2>
#  <p style='color:#888;margin-top:0;font-size:13px'>
#    Multi-Signal Trust Scoring Framework &nbsp;·&nbsp;
#    <em>Rajaram et al.</em> &nbsp;·&nbsp;
#    Powered by <b>Groq LLaMA 3.3 70B</b> + <b>Wikipedia</b> + <b>DuckDuckGo</b>
#    &nbsp;·&nbsp; <span style='color:#2ecc71'>Free — no billing required</span>
#  </p>
#</div>
#""")

# ── Assemble UI ───────────────────────────────────────────────────────────────
#ui = widgets.VBox([
#    header,
#    divider(),

#    H("📋 Step 1 — Enter the prompt and AI output to evaluate"),
#    prompt_box,
#    paste_box,
#    divider(),

#    H("⚠️  Step 2 — Set the task risk level"),
#    risk_slider,
#    risk_note,
#    divider(),

#    H("🎛️  Step 3 — Adjust signal weights  (renormalised automatically)"),
#    w_E, w_V, w_S, w_L, w_C,
#    signal_legend,
#    divider(),

#    routing_legend,
#    divider(),

#    widgets.HBox(
#        [run_btn, status_lbl],
#        layout=widgets.Layout(align_items='center', gap='12px')
#    ),
#    out,
#], layout=widgets.Layout(padding='16px', max_width='920px'))

#display(ui)



In [18]:
# ── Cell 8: Batch Evaluation + Heatmap ────────────────────────────────────────
#import pandas as pd

#def batch_evaluate(items: list, risk_level=None, weights=None) -> pd.DataFrame:
#    """
#    Evaluate a list of prompts.
#    items : list of dicts with key 'prompt' and optional key 'output'.
#    """
#    if risk_level is None: risk_level = DEFAULT_RISK_LEVEL
#    if weights    is None: weights    = DEFAULT_WEIGHTS
#    rows = []
#    for i, item in enumerate(items, 1):
#        print(f"[{i}/{len(items)}] {item['prompt'][:65]}")
#        res = evaluate_trust(item['prompt'], pasted_output=item.get('output'),
#                              risk_level=risk_level, weights=weights, verbose=False)
#        rows.append(res)
#        time.sleep(2)   # pause between items to respect rate limits

#    df = pd.DataFrame(rows)[['prompt','E','V','S','L','C','trust_score','routing','verdict']]

#    fig, ax = plt.subplots(figsize=(11, max(3, len(items)*0.7+1.2)))
#    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d26')
#    hdata = df[['E','V','S','L','C','trust_score']].values.copy()
#    hdata[:,-1] /= 100
#    im = ax.imshow(hdata, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
#    ax.set_xticks(range(6))
#    ax.set_xticklabels(['E','V','S','L','C','Trust/100'], color='white')
#    ax.set_yticks(range(len(df)))
#    ax.set_yticklabels([p[:45] for p in df['prompt']], color='white', fontsize=8)
#    for i in range(len(df)):
#        for j in range(6):
#            v = hdata[i,j]
#            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
#                    color='black' if 0.3<v<0.75 else 'white', fontsize=8)
#    plt.colorbar(im, ax=ax, label='Score (0–1)')
#    ax.set_title('Batch Trust Score Heatmap', color='white', fontsize=13)
#    plt.tight_layout(); plt.show()
#    return df

# ── Example (uncomment to run) ────────────────────────────────────────────────
#sample_batch = [
#     {"prompt": "What is the speed of light in a vacuum?"},
#     {"prompt": "Who invented the telephone?"},
#     {"prompt": "What year did World War II end?"},
# ]
#df_results = batch_evaluate(sample_batch)
#display(df_results)

#print("✅ Batch evaluation ready.")